In [1]:
# importar biliotecas

import pandas as pd
import geopandas as gpd
import numpy as np
import plotly
import plotly.express as px
import json
import os

In [2]:
# Caminho da pasta com os arquivos do Ceará

caminho_ceara = r"C:\Users\Lenovo\OneDrive\Área de Trabalho\BRAZIL JUSTICE LINKEDIN\justica_estadual\processos-tjce"


In [3]:
# Lista para guardar os DataFrames de cada arquivo

lista_dfs = []

In [4]:
for arquivo in os.listdir(caminho_ceara):
    if arquivo.endswith(".json"):
        caminho_arquivo = os.path.join(caminho_ceara, arquivo)
        try:
            with open(caminho_arquivo, encoding='utf-8') as f:
                dados = json.load(f)  # Carrega como dicionário
                df_temp = pd.json_normalize(dados)  # Converte dicionário aninhado em DataFrame plano
                lista_dfs.append(df_temp)
        except Exception as e:
            print(f"Erro ao carregar {arquivo}: {e}")

# Juntar todos os DataFrames
df_ceara = pd.concat(lista_dfs, ignore_index=True)


In [5]:
df_tjce = df_ceara[[
    'siglaTribunal',
    'grau',
    'dadosBasicos.valorCausa',
    'dadosBasicos.numero',
    'dadosBasicos.dataAjuizamento',
    'dadosBasicos.orgaoJulgador.nomeOrgao',
    'dadosBasicos.orgaoJulgador.codigoMunicipioIBGE',
    'dadosBasicos.orgaoJulgador.codigoOrgao',
    'dadosBasicos.orgaoJulgador.instancia',
    'dadosBasicos.competencia',
    'dadosBasicos.codigoLocalidade'
]]

In [6]:
# convertendo a coluna de data do ajuizamento para o formato datetime

df_tjce['dadosBasicos.dataAjuizamento'] = pd.to_datetime(
    df_tjce['dadosBasicos.dataAjuizamento'].astype(str), 
    format='%Y%m%d%H%M%S', 
    errors='coerce'  # transforma valores inválidos em NaT
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9200\3217021485.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tjce['dadosBasicos.dataAjuizamento'] = pd.to_datetime(


In [7]:
# removendo dados inválidos da coluna de data do ajuizamento

# Remover registros com ano menor que 1990

df_tjce = df_tjce[df_tjce['dadosBasicos.dataAjuizamento'].dt.year >= 1990]


In [8]:
# importando a relaçao de municípios e códigos disponibilizada pelo IBGE

arquivo = 'RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.xls'

df_municipios = pd.read_excel(arquivo, skiprows=6)


In [9]:
# Garantindo que os códigos de município estejam como string

df_tjce['dadosBasicos.orgaoJulgador.codigoMunicipioIBGE'] = df_tjce['dadosBasicos.orgaoJulgador.codigoMunicipioIBGE'].astype(str)
df_municipios['Código Município Completo'] = df_municipios['Código Município Completo'].astype(str)

In [10]:
# Fazendo o merge para incluir o nome do município 

df_tjce = df_tjce.merge(
    df_municipios[['Código Município Completo', 'Nome_Município']],
    how='left',
    left_on='dadosBasicos.orgaoJulgador.codigoMunicipioIBGE',
    right_on='Código Município Completo'
)

In [11]:
# Remover a coluna duplicada de código

df_tjce.drop(columns=['Código Município Completo'], inplace=True)

In [ ]:
# Renomeando colunas para uma visão mais limpa

df_tjce.rename(columns={
    'dadosBasicos.valorCausa': 'Valor da Causa',
    'dadosBasicos.numero': 'Numero',
    'dadosBasicos.dataAjuizamento': 'Data do Ajuizamento',
    'dadosBasicos.orgaoJulgador.nomeOrgao': 'Orgao Julgador',
    'dadosBasicos.orgaoJulgador.codigoMunicipioIBGE': 'Codigo do Municipio IBGE',
    'dadosBasicos.orgaoJulgador.codigoOrgao': 'Cod Orgao',
    'dadosBasicos.orgaoJulgador.instancia': 'Instancia',
    'dadosBasicos.competencia': 'Competencia',
    'dadosBasicos.codigoLocalidade': 'Cod Localidade'
}, inplace=True)

In [13]:
df_tjce.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54815 entries, 0 to 54814
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   siglaTribunal             54815 non-null  object        
 1   grau                      54815 non-null  object        
 2   Valor da Causa            54815 non-null  float64       
 3   Numero                    54815 non-null  object        
 4   Data do Ajuizamento       54815 non-null  datetime64[ns]
 5   Orgao Julgador            54815 non-null  object        
 6   Codigo do Municipio IBGE  54815 non-null  object        
 7   Cod Orgao                 54815 non-null  int64         
 8   Instancia                 54815 non-null  object        
 9   Competencia               54815 non-null  int64         
 10  Cod Localidade            54815 non-null  object        
 11  Nome_Município            54815 non-null  object        
dtypes: datetime64[ns](

In [14]:
################## INÍCIO DAS ANÁLISES ###################
################## INÍCIO DAS ANÁLISES ###################
################## INÍCIO DAS ANÁLISES ###################
################## INÍCIO DAS ANÁLISES ###################
################## INÍCIO DAS ANÁLISES ###################
################## INÍCIO DAS ANÁLISES ###################

In [15]:
# 1) Quais municípios concentram mais processos?

# Agrupar e ordenar
dados_agrupados = df_tjce.groupby('Nome_Município').size().reset_index(name='Quantidade')

# Excluir Fortaleza
dados_sem_fortaleza = dados_agrupados[dados_agrupados['Nome_Município'] != 'Fortaleza']

# Top 10 (sem Fortaleza)
dados_top10_sem_fortaleza = dados_sem_fortaleza.sort_values(by='Quantidade', ascending=False).head(10)

# Gráfico
fig = px.bar(
    dados_top10_sem_fortaleza,
    x='Nome_Município',
    y='Quantidade',
    title='🏙️ Municípios com mais processos no 1º Grau (Exceto Fortaleza - TJCE)',
    labels={'Nome_Município': 'Município', 'Quantidade': 'Número de Processos'},
    color_discrete_sequence=['#00429d'],
    )

fig.update_layout(bargap=0.5)

fig.show()

In [16]:
# 2) Quantidade de processos distribuidos em fortaleza em comparação a soma dos demais municípios

# Soma de processos de Fortaleza
qtd_fortaleza = df_tjce[df_tjce['Nome_Município'] == 'Fortaleza'].shape[0]

# Top 10 municípios com mais processos, excluindo Fortaleza
top10_sem_fortaleza = (
    df_tjce[df_tjce['Nome_Município'] != 'Fortaleza']
    .groupby('Nome_Município')
    .size()
    .sort_values(ascending=False)
    .head(10)
)

# Soma total dos processos desses 10 municípios
qtd_outros_10 = top10_sem_fortaleza.sum()

# Criando DataFrame para o gráfico
comparacao = pd.DataFrame({
    'Município': ['Fortaleza', 'Top 10 sem Fortaleza (soma)'],
    'Quantidade de Processos': [qtd_fortaleza, qtd_outros_10]
})

# Gráfico de barras comparativo
fig = px.bar(
    comparacao,
    x='Município',
    y='Quantidade de Processos',
    title='📊 Comparativo: Fortaleza x Top 10 Municípios (sem Fortaleza)',
    text='Quantidade de Processos',
    color='Município',
    color_discrete_sequence=[ '#ff7f0e', '#00429d']
)

fig.update_layout(
    xaxis_title='',
    yaxis_title='Número de Processos',
    bargap=0.4,
    showlegend=False
)

fig.update_layout(bargap=0.6)


fig.show()

In [38]:
# 2) Órgãos julgadores com maiores demandas

# Selecionar os 10 municípios com mais processos
top_municipios = df_tjce['Nome_Município'].value_counts().head(10).index.tolist()

# Filtrar apenas os dados desses municípios
df_top = df_tjce[df_tjce['Nome_Município'].isin(top_municipios)]

# Agrupar por município e órgão julgador, e contar os processos
grupo = df_top.groupby(['Nome_Município', 'Orgao Julgador']).size().reset_index(name='quantidade')

# Para cada município, manter apenas o órgão com maior número de processos
orgao_maior_demanda = grupo.sort_values('quantidade', ascending=False).drop_duplicates(subset=['Nome_Município'])

# Ordenar os municípios pelo total de processos para melhor visualização
orgao_maior_demanda = orgao_maior_demanda.sort_values('quantidade', ascending=False)

# Plotar o gráfico de barras
import plotly.express as px

fig = px.bar(orgao_maior_demanda,
             x='Nome_Município',
             y='quantidade',
             color='Orgao Julgador',
             text='quantidade',
             title='Órgãos Julgadores com Maior Demanda por Município (Top 10 Municípios)',
             labels={
                 'quantidade': 'Qtd. de Processos',
                 'Nome_Município': 'Município',
                 'Orgao Julgador': 'Órgão Julgador'
             })
fig.update_layout(uniformtext_minsize=8, uniformtext_mode='show',margin=dict(t=100))
fig.update_traces(textposition='inside')
fig.show()


In [47]:
# 3) Processos com valor da causa superior a ... (ex: R$ 1 milhão)

pd.options.display.float_format = '{:,.2f}'.format
df_tjce.sort_values(by='Valor da Causa', ascending=False).head(10)



,siglaTribunal,grau,Valor da Causa,Numero,Data do Ajuizamento,Orgao Julgador,Codigo do Municipio IBGE,Cod Orgao,Instancia,Competencia,Cod Localidade,Nome_Município,Valor Formatado
19355,TJCE,G1,"87,539,910,942.23",00239482920008060001,1993-03-29,2� VARA CIVEL DA COMARCA DE FORTALEZA,2304400,8414,ORIG,115,2304400,Fortaleza,"R$ 87.539.910.942,23"
19513,TJCE,G1,"84,488,317,788.70",01798264420008060001,1993-11-03,2� VARA DE EXECU�OES FISCAIS DA COMARCA DE FOR...,2304400,8648,ORIG,44,2304400,Fortaleza,"R$ 84.488.317.788,70"
19373,TJCE,G1,"84,488,317,758.70",01798109020008060001,1993-05-17,2� VARA DE EXECU�OES FISCAIS DA COMARCA DE FOR...,2304400,8648,ORIG,44,2304400,Fortaleza,"R$ 84.488.317.758,70"
19384,TJCE,G1,"35,915,158,462.46",00138585920008060001,1993-06-09,2� VARA CIVEL DA COMARCA DE FORTALEZA,2304400,8414,ORIG,115,2304400,Fortaleza,"R$ 35.915.158.462,46"
18903,TJCE,G1,"8,207,203,820.75",02094694720008060001,1992-04-02,9ª VARA CIVEL DA COMARCA DE FORTALEZA,2304400,8451,ORIG,2,2304400,Fortaleza,"R$ 8.207.203.820,75"
13266,TJCE,G2,"4,199,938,354.00",06233060920208060000,2020-03-23,3� CAMARA CRIMINAL,2304400,81560,ORIG,-3,2304400,Fortaleza,"R$ 4.199.938.354,00"
39592,TJCE,G1,"3,000,292,006.06",00040572420108060081,2010-07-28,2� VARA DA COMARCA DE GRANJA,2304707,8516,ORIG,118,2304707,Granja,"R$ 3.000.292.006,06"
18891,TJCE,G1,"2,622,047,605.25",01674770920008060001,1992-02-28,2� VARA CIVEL DA COMARCA DE FORTALEZA,2304400,8414,ORIG,115,2304400,Fortaleza,"R$ 2.622.047.605,25"
19710,TJCE,G1,"1,698,268,743.00",00240383720008060001,1994-05-02,2� VARA CIVEL DA COMARCA DE FORTALEZA,2304400,8414,ORIG,115,2304400,Fortaleza,"R$ 1.698.268.743,00"
19362,TJCE,G1,"1,330,000,000.00",00520497620008060001,1993-05-03,9� VARA CIVEL DA COMARCA DE FORTALEZA,2304400,8451,ORIG,115,2304400,Fortaleza,"R$ 1.330.000.000,00"


In [ ]:
# 4) Acho que vai dar um trabalhinho. Mas a quantidade de processos distribuidos por decada (de 1990 a 2000, de 2000 a 2010, de 2010 a 2020)

In [21]:
#Post com storytelling: “O que aprendi ao analisar 54 mil processos do TJCE”

#Conte como usou Python, merge, plotly, etc.

#Mostre 2 ou 3 achados

#Finalize com uma pergunta ou convite ao diálogo